In [1]:
import torch
import numpy as np
import os
import matplotlib.pyplot as plt

from config import Config
from data_generator import SimpleDataGenerator
from model import SimpleMultiWavelengthModel
from trainer import SimpleTrainer
from visualizer import SimpleVisualizer
from propagation import angular_spectrum_propagation

def main():
    # 设置随机种子，确保结果可复现
    torch.manual_seed(42)
    np.random.seed(42)
    
    # 创建配置对象
    config = Config(
        num_modes=1,
        wavelengths=[450e-9, 650e-9],  # 蓝光和红光
        field_size=100,
        layer_size=200,
        pixel_size=1.0e-6,  # 1微米
        num_epochs=300,
        save_dir="./results/",
        offsets=[(10, 10), (-10, -10)],  # 不同波长的目标位置偏移
        detect_size=10  # 检测区域大小
    )
    
    # 创建数据生成器
    data_generator = SimpleDataGenerator(config)
    
    # 创建训练器
    trainer = SimpleTrainer(config, data_generator)
    
    # 创建可视化器
    visualizer = SimpleVisualizer(config)
    
    # 创建模型
    model = SimpleMultiWavelengthModel(config, num_layers=2).to(config.device)
    
    # 训练模型
    training_results = trainer.train_model(model)
    
    # 获取训练后的模型
    trained_model = training_results['model']
    
    # 生成输入场
    input_fields = data_generator.generate_input_fields()
    
    # 获取训练后的输出场
    output_fields = trained_model(input_fields)
    
    # 可视化训练损失
    visualizer.plot_loss_curve(training_results['losses'], 
                              f"模型训练损失曲线 (层数: {trained_model.num_layers})",
                              f"{config.save_dir}/loss_curve.png")
    
    # 可视化能量分布
    visualizer.plot_energy_distributions(
        output_fields, 
        config.wavelengths,
        f"{config.save_dir}/energy_distributions.png"
    )
    
    # 可视化相位掩膜
    visualizer.plot_phase_masks(
        [param.detach() for param in trained_model.phase_masks],
        f"{config.save_dir}/phase_masks.png"
    )
    
    # 可视化光场传播过程
    # 获取传播过程中的场
    propagation_fields = []
    
    # 添加初始场
    current_fields = input_fields.copy()
    propagation_fields.append([field.clone() for field in current_fields])
    
    # 对每个相位掩膜进行传播
    for layer_idx in range(trained_model.num_layers):
        # 应用相位掩膜
        phase_mask = trained_model.phase_masks[layer_idx]
        for w_idx in range(len(current_fields)):
            # 应用相位掩膜
            amplitude = torch.abs(current_fields[w_idx])
            phase_shift = torch.exp(1j * phase_mask)
            current_fields[w_idx] = amplitude * phase_shift
            
            # 传播到下一层
            current_fields[w_idx] = angular_spectrum_propagation(
                current_fields[w_idx],
                trained_model.propagation_distances[layer_idx],
                config.wavelengths[w_idx],
                config.pixel_size
            )
        
        # 保存当前传播状态
        propagation_fields.append([field.clone() for field in current_fields])
    
    # 为每个波长可视化传播过程
    for w_idx, wavelength in enumerate(config.wavelengths):
        # 提取该波长的所有传播场
        wavelength_fields = [fields[w_idx] for fields in propagation_fields]
        
        # 可视化传播过程
        visualizer.plot_field_propagation(
            wavelength_fields,
            f"波长 {int(wavelength*1e9)}nm 的光场传播过程",
            f"{config.save_dir}/propagation_wavelength_{int(wavelength*1e9)}nm.png"
        )
    
    # 可视化最终目标区域的能量分布
    visualizer.plot_target_regions(
        output_fields,
        config.wavelengths,
        config.offsets,
        config.detect_size,
        f"{config.save_dir}/target_regions.png"
    )
    
    # 计算并显示波长分离性能指标
    separation_metrics = trainer.calculate_wavelength_separation(trained_model)
    print("波长分离性能指标:")
    for wavelength, metrics in separation_metrics.items():
        print(f"波长 {int(wavelength*1e9)}nm:")
        print(f"  - 目标区域能量比例: {metrics['target_ratio']:.4f}")
        print(f"  - 对比度: {metrics['contrast']:.4f}")
    
    # 可视化波长分离性能
    visualizer.plot_wavelength_separation_metrics(
        separation_metrics,
        f"{config.save_dir}/separation_metrics.png"
    )
    
    print("可视化结果已保存到:", config.save_dir)

if __name__ == "__main__":
    main()


/home/shiyue/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch [50/300], Loss: -19.019049
Epoch [100/300], Loss: -22.117611
Epoch [150/300], Loss: -22.024200
Epoch [200/300], Loss: -23.024452
Epoch [250/300], Loss: -23.895908
Epoch [300/300], Loss: -23.135124
Training completed in 2.03 seconds.


TypeError: SimpleTrainer.calculate_visibility() missing 1 required positional argument: 'input_fields'